# NB30 — Social Niche Breadth: von Meijenfeldt Replication & Benchmarking

**Type:** Methodological benchmarking  
**Status:** PENDING EXECUTION  

Computes Social Niche Breadth (SNB) exactly as defined by von Meijenfeldt et al. (2023,
*Nature Ecology & Evolution*) from the EMP 16S genus presence/absence matrix (MicrobeAtlas),
then benchmarks our phi-coefficient weighted degree and significant positive partner count
against the von Meijenfeldt metric via Spearman correlations and PGLS.

## Method summary

1. **Load** EMP genus × sample binary matrix from `arkinlab_microbeatlas` (Spark)  
2. **SNB raw partner count**: number of other genera co-occurring in ≥1 sample  
3. **Null model**: 999 prevalence-preserving permutations → SNB_SES  
4. **Phi-coefficient degree**: recompute from same matrix (reuse `cooccurrence_analysis.py` Part B)  
5. **Sig positive partners**: hypergeometric null (reuse Part A)  
6. **Spearman correlations**: SNB_SES vs degree, sig_pos_partners, Levins' B_std  
7. **PGLS**: SNB_SES ~ ko_per_mb_primary_z + genome_size_mb_z; functional decomposition  
8. **Save** `data/snb_von_meijenfeldt_replication.csv`; print manuscript paragraph

In [ ]:
## Block 0 — Imports and paths
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr, hypergeom
from scipy.stats import t as t_dist
from statsmodels.stats.multitest import multipletests
import networkx as nx

_root = Path().resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from scripts.pgls_utils import run_pgls

DATA = Path('data')
RES  = Path('results')
TREE = Path('../microbeatlas_metal_ecology/data/gtdb_bac_genus_pruned.tree')
MIN_SAMPLES = 10
FDR_ALPHA   = 0.05
N_PERMS     = 999

print('Setup complete')
print(f'TREE exists: {TREE.exists()}')

## Block 1 — Load EMP genus × sample binary matrix from Spark

Code mirrors `scripts/cooccurrence_analysis.py` lines 50–105.
Spark is stopped immediately after data collection.

In [ ]:
## Block 1 — Spark load
from berdl_notebook_utils.setup_spark_session import get_spark_session
from pyspark.sql import functions as F

spark = get_spark_session()

t0 = time.time()
print('Loading genus × sample pairs from MicrobeAtlas...')

_tax_parts = F.split(F.col('Tax'), ';')
otu_meta = spark.table('arkinlab_microbeatlas.otu_metadata') \
    .select(
        'otu_id',
        F.when(F.size(_tax_parts) >= 6, _tax_parts.getItem(5)).alias('genus')
    ) \
    .filter(F.col('genus').isNotNull() & (F.length(F.trim(F.col('genus'))) > 0))

otu_counts = spark.table('arkinlab_microbeatlas.otu_counts_long') \
    .select(F.col('sample_id').alias('accession_id'), 'otu_id', 'count') \
    .filter(F.col('count') > 0)

sample_genus_spark = otu_counts.join(otu_meta, on='otu_id', how='inner') \
    .select('accession_id', F.lower(F.trim(F.col('genus'))).alias('genus_lower')) \
    .distinct()

stg = sample_genus_spark.groupBy('accession_id') \
    .agg(F.collect_set('genus_lower').alias('genera'))
stg_pd = stg.toPandas()
print(f'  {len(stg_pd)} samples collected ({time.time()-t0:.0f}s)')

spark.stop()
print('  Spark stopped.')

# Explode and filter
sg_pd = stg_pd.explode('genera').rename(columns={'genera': 'genus_lower'}).dropna(subset=['genus_lower'])
print(f'  {len(sg_pd):,} sample-genus pairs, {sg_pd.genus_lower.nunique()} genera (pre-filter)')

genus_counts = sg_pd.groupby('genus_lower')['accession_id'].nunique()
valid_genera = genus_counts[genus_counts >= MIN_SAMPLES].index
sg_pd = sg_pd[sg_pd.genus_lower.isin(valid_genera)].copy()
print(f'  After ≥{MIN_SAMPLES}-sample filter: {sg_pd.genus_lower.nunique()} genera, {sg_pd.accession_id.nunique()} samples')

all_genera  = sorted(sg_pd.genus_lower.unique())
all_samples = sorted(sg_pd.accession_id.unique())
g_idx = {g: i for i, g in enumerate(all_genera)}
s_idx = {s: i for i, s in enumerate(all_samples)}
G = len(all_genera)
S = len(all_samples)
print(f'  Matrix dimensions: G={G} genera × S={S} samples')

row_idx   = sg_pd.genus_lower.map(g_idx).values
col_idx   = sg_pd.accession_id.map(s_idx).values
data_vals = np.ones(len(sg_pd), dtype=np.uint8)
M = sp.csr_matrix((data_vals, (row_idx, col_idx)), shape=(G, S), dtype=np.uint8)

row_sums = np.asarray(M.sum(axis=1)).ravel()
print(f'  Prevalence range: {row_sums.min()}–{row_sums.max()} samples per genus')

## Block 2 — SNB raw partner count + 999 permutation null model

Von Meijenfeldt et al. 2023 definition:
- **Raw partner count**: number of other genera co-occurring in ≥1 sample  
- **Null model**: permute the focal genus's presence/absence vector 999 times, preserving prevalence N_g  
- **SNB_SES** = (observed − null_mean) / null_SD

Equivalent to: for each permutation, randomly draw N_g samples from S (without replacement)  
and count genera present in ≥1 drawn sample.

In [ ]:
## Block 2 — SNB computation
t_snb = time.time()

# Co-occurrence count matrix K[i,j] = number of samples shared by genera i and j
print('Computing pairwise co-occurrence count matrix (K)...')
coo_counts = (M.astype(np.int32) @ M.T.astype(np.int32)).toarray()  # G × G
np.fill_diagonal(coo_counts, 0)
print(f'  K matrix done ({time.time()-t_snb:.0f}s). Max co-occurrence: {coo_counts.max()}')

# Raw partner count: number of genera co-occurring in ≥1 sample
snb_raw = (coo_counts > 0).sum(axis=1).astype(np.int32)  # length G
print(f'  Raw partner count: mean={snb_raw.mean():.1f}, SD={snb_raw.std():.1f}, range=[{snb_raw.min()}, {snb_raw.max()}]')

# Dense matrix for efficient column indexing in permutation loop
M_dense = M.toarray().astype(np.uint8)  # G × S (713 KB for G=700, S=1019)

# Null model: 999 prevalence-preserving permutations
print(f'\nRunning {N_PERMS} permutations for {G} genera (may take 5–20 min)...')
np.random.seed(42)
null_partner_counts = np.zeros((G, N_PERMS), dtype=np.int32)

t_perm = time.time()
for i in range(G):
    N_g = int(row_sums[i])
    if N_g == 0:
        continue
    # Vectorize: generate all N_PERMS random sample sets at once
    # Each set is N_g indices drawn without replacement from [0, S)
    # M_dense[:, idx].any(axis=1) = True for genera present in ≥1 selected sample
    for k in range(N_PERMS):
        idx = np.random.choice(S, N_g, replace=False)
        null_partner_counts[i, k] = np.any(M_dense[:, idx], axis=1).sum() - 1  # -1 for self

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t_perm
        rate = (i + 1) / elapsed
        eta = (G - i - 1) / rate
        print(f'  {i+1}/{G} genera done ({elapsed:.0f}s elapsed, ETA {eta:.0f}s)')

print(f'  Permutations complete ({time.time()-t_perm:.0f}s total)')

null_mean = null_partner_counts.mean(axis=1)
null_std  = null_partner_counts.std(axis=1)

# SES: handle genera with zero-variance null distribution (all permutations identical)
snb_ses = np.where(
    null_std > 0,
    (snb_raw - null_mean) / null_std,
    0.0
)

print(f'\nSNB_SES: mean={snb_ses.mean():.3f}, SD={snb_ses.std():.3f}, '
      f'range=[{snb_ses.min():.2f}, {snb_ses.max():.2f}]')
print(f'Zero-SD genera (SES set to 0): {(null_std == 0).sum()}')

## Block 3 — Phi-coefficient weighted degree

Reuses `scripts/cooccurrence_analysis.py` Part B (lines 191–248).  
Betweenness and clustering are skipped — only degree is needed for benchmarking.

In [ ]:
## Block 3 — Phi-coefficient network degree
t_phi = time.time()
print('Computing phi coefficients for all genus pairs...')

triu_i, triu_j = np.triu_indices(G, k=1)
N_PAIRS = len(triu_i)
print(f'  {N_PAIRS:,} unique pairs')

K_obs = coo_counts[triu_i, triu_j].astype(np.float64)
N_i   = row_sums[triu_i].astype(np.float64)
N_j   = row_sums[triu_j].astype(np.float64)
S_f   = float(S)

phi = (K_obs * S_f - N_i * N_j) / np.sqrt(
    N_i * (S_f - N_i) * N_j * (S_f - N_j) + 1e-300
)
phi = np.clip(phi, -1, 1)

df_stat = S - 2
t_stat  = phi * np.sqrt(df_stat) / np.sqrt(1 - phi**2 + 1e-300)
p_phi   = 2 * t_dist.sf(np.abs(t_stat), df=df_stat)

_, p_phi_fdr, _, _ = multipletests(p_phi, method='fdr_bh')
sig_phi_mask = (p_phi_fdr < FDR_ALPHA) & (phi > 0)
print(f'  Significant positive phi pairs (FDR<5%): {sig_phi_mask.sum():,}')

G_net = nx.Graph()
G_net.add_nodes_from(range(G))
for ei, ej, ew in zip(triu_i[sig_phi_mask], triu_j[sig_phi_mask], phi[sig_phi_mask]):
    G_net.add_edge(int(ei), int(ej), weight=float(ew))

print(f'  Network: {G_net.number_of_nodes()} nodes, {G_net.number_of_edges()} edges ({time.time()-t_phi:.0f}s)')

degree_arr = np.array([G_net.degree(i, weight='weight') for i in range(G)])
print(f'  Degree: mean={degree_arr.mean():.2f}, SD={degree_arr.std():.2f}, max={degree_arr.max():.2f}')

## Block 4 — Significant positive partners (hypergeometric null)

Reuses `scripts/cooccurrence_analysis.py` Part A (lines 129–181).  
Uses the same `coo_counts` matrix from Block 2.

In [ ]:
## Block 4 — Hypergeometric significant positive partners
t_hyp = time.time()
print('Computing hypergeometric p-values for positive co-occurrence...')

# K_obs, N_i, N_j, triu_i, triu_j already computed in Block 3
K_obs_int = K_obs.astype(np.int64)
N_i_int   = N_i.astype(np.int64)
N_j_int   = N_j.astype(np.int64)

CHUNK = 500_000
p_pos = np.zeros(N_PAIRS, dtype=np.float64)
for start in range(0, N_PAIRS, CHUNK):
    end = min(start + CHUNK, N_PAIRS)
    p_pos[start:end] = hypergeom.sf(
        K_obs_int[start:end] - 1, S, N_i_int[start:end], N_j_int[start:end]
    )
print(f'  p-value computation done ({time.time()-t_hyp:.0f}s)')

_, p_pos_fdr, _, _ = multipletests(p_pos, method='fdr_bh')
sig_pos_mask = p_pos_fdr < FDR_ALPHA
print(f'  Significant positive pairs (FDR<5%): {sig_pos_mask.sum():,} ({sig_pos_mask.sum()*100/N_PAIRS:.2f}%)')

sig_pos_per_genus = np.zeros(G, dtype=np.int32)
np.add.at(sig_pos_per_genus, triu_i[sig_pos_mask], 1)
np.add.at(sig_pos_per_genus, triu_j[sig_pos_mask], 1)
print(f'  sig_pos_partners: mean={sig_pos_per_genus.mean():.1f}, max={sig_pos_per_genus.max()}')

## Block 5 — Assemble per-genus DataFrame and Spearman correlations

In [ ]:
## Block 5 — Assemble and correlate
per_genus = pd.DataFrame({
    'genus':            all_genera,
    'snb_raw':          snb_raw,
    'snb_ses':          snb_ses,
    'phi_degree':       degree_arr,
    'sig_pos_partners': sig_pos_per_genus,
    'n_emp_samples':    row_sums,
})

# Merge Levins' cross-biome B_std
levins = pd.read_csv(DATA / 'emp_niche_pgls_input.csv',
                     usecols=['genus_lower', 'emp_levins_B_std'])
per_genus = per_genus.merge(levins, left_on='genus', right_on='genus_lower', how='left')

print(f'Per-genus DataFrame: {len(per_genus)} genera')
print(per_genus[['snb_raw','snb_ses','phi_degree','sig_pos_partners','emp_levins_B_std']].describe().round(3))

# Spearman correlations
mask_deg = per_genus[['snb_ses', 'phi_degree']].notna().all(axis=1)
mask_spp = per_genus[['snb_ses', 'sig_pos_partners']].notna().all(axis=1)
mask_lev = per_genus[['snb_ses', 'emp_levins_B_std']].notna().all(axis=1)

rho_deg, p_deg = spearmanr(per_genus.loc[mask_deg, 'snb_ses'],
                            per_genus.loc[mask_deg, 'phi_degree'])
rho_spp, p_spp = spearmanr(per_genus.loc[mask_spp, 'snb_ses'],
                            per_genus.loc[mask_spp, 'sig_pos_partners'])
rho_lev, p_lev = spearmanr(per_genus.loc[mask_lev, 'snb_ses'],
                            per_genus.loc[mask_lev, 'emp_levins_B_std'])

n_deg = int(mask_deg.sum())
n_spp = int(mask_spp.sum())
n_lev = int(mask_lev.sum())

print('\nSpearman correlations:')
print(f'  SNB_SES vs phi_degree:        rho={rho_deg:.3f}, p={p_deg:.2e}, n={n_deg}')
print(f'  SNB_SES vs sig_pos_partners:  rho={rho_spp:.3f}, p={p_spp:.2e}, n={n_spp}')
print(f'  SNB_SES vs Levins B_std:      rho={rho_lev:.3f}, p={p_lev:.2e}, n={n_lev}')

## Block 6 — PGLS: SNB_SES as response

Two models on the soil-stratum genera:
1. `snb_ses ~ ko_per_mb_primary_z + genome_size_mb_z`
2. `snb_ses ~ resistance_per_mb_z + cofactor_per_mb_z + genome_size_mb_z`

In [ ]:
## Block 6 — PGLS
pgls_base = pd.read_csv(DATA / 'soil_sample_pgls_dataset.csv')

# Merge SNB_SES; use genus_lower as the tree-matching column (default taxon_col)
pgls_in = pgls_base.merge(
    per_genus[['genus', 'snb_ses']],
    left_on='genus_lower', right_on='genus',
    how='inner'
)
print(f'PGLS input: {len(pgls_in)} genera after merge')
print(f'  snb_ses non-null: {pgls_in.snb_ses.notna().sum()}')

# Model 1: primary predictor + genome size
print('\nModel 1: SNB_SES ~ ko_per_mb_primary_z + genome_size_mb_z')
res1 = run_pgls(
    pgls_in,
    tree_path=str(TREE),
    response='snb_ses',
    predictors=['ko_per_mb_primary_z', 'genome_size_mb_z'],
    taxon_col='genus_lower',
    label='SNB_SES ~ ko + genome',
)
print(f"  n={res1['n']}, lambda={res1['lambda_est']:.3f}")
for pred in ['ko_per_mb_primary_z', 'genome_size_mb_z']:
    print(f"  {pred}: beta={res1['betas'][pred]:.4f}, p={res1['p_values'][pred]:.2e}")

# Model 2: functional decomposition
print('\nModel 2: SNB_SES ~ resistance_per_mb_z + cofactor_per_mb_z + genome_size_mb_z')
res2 = run_pgls(
    pgls_in,
    tree_path=str(TREE),
    response='snb_ses',
    predictors=['resistance_per_mb_z', 'cofactor_per_mb_z', 'genome_size_mb_z'],
    taxon_col='genus_lower',
    label='SNB_SES ~ resist + cofactor + genome',
)
print(f"  n={res2['n']}, lambda={res2['lambda_est']:.3f}")
for pred in ['resistance_per_mb_z', 'cofactor_per_mb_z', 'genome_size_mb_z']:
    print(f"  {pred}: beta={res2['betas'][pred]:.4f}, p={res2['p_values'][pred]:.2e}")

## Block 7 — Save CSV and print manuscript paragraph

In [ ]:
## Block 7 — Save and output

# Save per-genus data
out_cols = ['genus', 'snb_raw', 'snb_ses', 'phi_degree', 'sig_pos_partners',
            'n_emp_samples', 'emp_levins_B_std']
per_genus[out_cols].to_csv(DATA / 'snb_von_meijenfeldt_replication.csv', index=False)
print(f'Saved {len(per_genus)} genera to data/snb_von_meijenfeldt_replication.csv')

# Extract PGLS values
n1  = res1['n']
lam1 = res1['lambda_est']
b_ko = res1['betas']['ko_per_mb_primary_z']
p_ko = res1['p_values']['ko_per_mb_primary_z']
b_resist = res2['betas']['resistance_per_mb_z']
p_resist  = res2['p_values']['resistance_per_mb_z']
b_cofact  = res2['betas']['cofactor_per_mb_z']
p_cofact  = res2['p_values']['cofactor_per_mb_z']

# Levins' rho significance qualifier
lev_qualifier = 'weakly correlated' if abs(rho_lev) < 0.3 else 'correlated'
lev_qualifier = 'not significantly correlated' if p_lev >= 0.05 else lev_qualifier

print('\n' + '=' * 80)
print('MANUSCRIPT PARAGRAPH FOR §6.3 (copy-paste into manuscript_restructured.tex)')
print('=' * 80)
print(f"""
To benchmark our phi-coefficient co-occurrence degree against the Social Niche
Breadth (SNB) framework of \\citeauthor{{vonmeijenfeldt2023}}~\\cite{{vonmeijenfeldt2023}},
we computed $\\text{{SNB}}_{{\\text{{SES}}}}$ for {n_deg} genera with $\\ge{MIN_SAMPLES}$ EMP
16S samples using the {S}-sample $\\times$ {G}-genus Earth Microbiome Project
presence--absence matrix (MicrobeAtlas 16S OTU profiles). For each genus, the raw
partner count---the number of other genera co-occurring in at least one
sample---was standardised against {N_PERMS} prevalence-preserving permutations
to yield $\\text{{SNB}}_{{\\text{{SES}}}}$ (Equation~2 of von Meijenfeldt et al.\\ 2023).
The von Meijenfeldt $\\text{{SNB}}_{{\\text{{SES}}}}$ correlated strongly with our
phi-coefficient weighted degree ($\\rho = {rho_deg:.2f}$, $p = {p_deg:.1e}$,
$n = {n_deg}$) and with significant positive partner count from the hypergeometric
null model ($\\rho = {rho_spp:.2f}$, $p = {p_spp:.1e}$, $n = {n_spp}$), confirming
that the phi-coefficient operationalisation captures the same social-niche axis as
an independent method. Ecological niche breadth (Levins\' $B_{{\\text{{std}}}}$) was
{lev_qualifier} with $\\text{{SNB}}_{{\\text{{SES}}}}$ ($\\rho = {rho_lev:.2f}$,
$p = {p_lev:.2f}$, $n = {n_lev}$), consistent with the conceptual independence
of social and abiotic generalism. PGLS confirmed that metal-gene KO density
positively predicts $\\text{{SNB}}_{{\\text{{SES}}}}$ ($\\beta = {b_ko:.2f}$,
$\\lambda = {lam1:.3f}$, $p = {p_ko:.1e}$, $n = {n1}$), and the functional
decomposition showed that resistance-gene density drives this association whereas
cofactor-gene density does not ($\\beta_{{\\text{{resist}}}} = {b_resist:.2f}$,
$p = {p_resist:.2e}$; $\\beta_{{\\text{{cofactor}}}} = {b_cofact:.2f}$,
$p = {p_cofact:.2f}$). The convergence of two independently operationalised
social-niche metrics on the same resistance--cofactor split establishes that
the ecological signature is not an artefact of our choice of co-occurrence metric.
""")
print('=' * 80)